In [ ]:
import torch
import torchvision
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import numpy as np
from scipy.stats import truncnorm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import image as img
from sklearn.model_selection import train_test_split
import matplotlib.patches as patches
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
%matplotlib notebook

# 产生训练数据
## 定义产生数据的函数

In [ ]:
def atrous(image):
    image_nan_to_0 = np.float32(np.nan_to_num(image, nan=0.0))
    coe = img.a_trous(image_nan_to_0, level=4)
    hq = coe.data[1] + coe.data[2]
    # hq = img.box_car_smoothing(hq, kernel_size=7)
    hq[np.isnan(image)] = np.nan
    return hq

def unsharp_mask(image):
    image_nan_to_0 = np.float32(np.nan_to_num(image, nan=0.0))
    unsharp_mask_image = img.unsharp_mask(image_nan_to_0, kernel_size=21, high_freq_only=True)
    unsharp_mask_image[np.isnan(image)] = np.nan
    return unsharp_mask_image

def normalization(data):
    mean = np.mean(data)
    std = np.std(data)
    return (data - mean) / std

def simulation(x, t, dx=120, dt=3, num_range=(100, 200), seed=None):
    if seed is not None:
        np.random.seed(seed)

    life_span = {'mean': 60, 'std': 30} # s
    brightness = {'mean': 0.2, 'std': 3.85}
    period = {'min': 60, 'max': 120} # s
    amplitude = {'mean': 121, 'std': 3} # km/s
    width = 9
    noise = {'mean': 0, 'std': 0.66}

    x_vect = np.arange(0, x, dx)
    t_vect = np.arange(0, t, dt)

    space_time = np.zeros((len(t_vect), len(x_vect)))

    num = int(np.random.uniform(num_range[0], num_range[-1]))
    boxes = []
    labels = []

    for i in range(num):
        life = truncnorm.rvs(-1, 3, loc=life_span['mean'], scale=life_span['std'])
        phase = np.random.uniform(0, 2*np.pi)
        T = np.random.uniform(period['min'], period['max'])
        I = truncnorm.rvs(1, 5, loc=brightness['mean'], scale=brightness['std'])
        speed = truncnorm.rvs(-3, 3, loc=amplitude['mean'], scale=amplitude['std'])
        A = 0.5*np.cos(np.random.uniform(0, 0.5*np.pi))*speed*T/np.pi
        begin_time = np.random.uniform(0, t-int(life))
        begin_x = np.random.uniform(int(A), x-int(A))

        t_index = (np.arange(begin_time, begin_time+life, dt)/dt).astype(int)
        t_index = np.clip(t_index, 0, len(t_vect)-1)
        c_index = np.floor((begin_x + A*np.sin(2*np.pi * t_index*dt/T + phase) + np.random.normal(0, A/5, len(t_index)))/dx).astype(int)
        c_index = np.clip(c_index, 0, len(x_vect)-1)

        x_indexes = []

        for j, time in enumerate(t_index):
            c = c_index[j]
            x_index = (np.array(range(width)) - width//2 + c).astype(int)
            x_index = np.clip(x_index, 0, len(x_vect)-1)
            x_indexes.extend(x_index)
            for index in x_index:
                space_time[time, index] = I * np.exp(-(index-c)**2 / (2 * (width/12)**2))
        x_indexes = np.array(x_indexes)
        box = [np.min(x_indexes), np.min(t_index), np.max(x_indexes), np.max(t_index)]
        boxes.append(box)
        labels.extend([1])

    space_time += truncnorm.rvs(-10, 10, loc=noise['mean'], scale=noise['std'], size=(len(t_vect), len(x_vect)))
    target = {
        'boxes': np.array(boxes),
        'labels': np.array(labels, dtype=np.int64),
    }
    return normalization(space_time), target

## 定义一个类用于存储所有数据

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, images, targets):
        self.images = images
        self.targets = targets

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index]
        target = self.targets[index]
        image_tensor = torch.tensor(image, dtype=torch.float).unsqueeze(0).repeat(3,1,1)
        boxes_tensor = torch.tensor(target['boxes'], dtype=torch.float32)
        labels_tensor = torch.tensor(target['labels'], dtype=torch.int64)
        target_tensor = {
            'boxes': boxes_tensor,
            'labels': labels_tensor,
        }

        return image_tensor, target_tensor

def collate_fn(batch):
    return tuple(zip(*batch))

# 划分数据集

In [ ]:
def split_dataset(images, targets, train=0.8, validation=0.1, test=0.1, random_state=None):
    if abs(train + validation + test - 1.0) > 1e-6:
        raise ValueError("train + validation + test must equal 1")

    # 先划分训练集与临时集
    train_images, temp_images, train_targets, temp_targets = train_test_split(
        images, targets, test_size=1-train, random_state=random_state)
    # 再将临时集划分为验证集与测试集
    val_images, test_images, val_targets, test_targets = train_test_split(
        temp_images, temp_targets, test_size=test/(validation+test), random_state=random_state)

    train_dataset = CustomDataset(train_images, train_targets)
    val_dataset = CustomDataset(val_images, val_targets)
    test_dataset = CustomDataset(test_images, test_targets)

    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

    return train_loader, val_loader, test_loader

# 训练模型

In [ ]:
def train_model(train_loader, val_loader, epochs=20, plot=True):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(num_classes=2, min_size=320, image_mean=[0, 0, 0], image_std=[1, 1, 1])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = optim.SGD(model.parameters(), lr=1e-4, momentum=0.9, weight_decay=1e-4)
    model.to(device)
    best_val_loss = float('inf')
    train_losses = []
    val_losses = []

    for epoch in tqdm(range(epochs), desc="training", total=epochs):
        # 在训练集上训练模型
        model.train()
        epoch_loss = 0

        for images_batch, targets_batch in train_loader:
            # images_batch 和 targets_batch 均为元组，转换为列表并移到 device 上
            images_batch = [img.to(device) for img in images_batch]
            targets_batch = [{k: v.to(device) for k, v in t.items()} for t in targets_batch]

            optimizer.zero_grad()

            loss_dict = model(images_batch, targets_batch)
            losses = sum(loss for loss in loss_dict.values())
            losses.backward()
            optimizer.step()
            epoch_loss += losses.item()

        avg_train_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # 在验证集上评估模型
        model.train()
        val_loss = 0.0
        with torch.no_grad():
            for images_batch, targets_batch in val_loader:
                images_batch = [img.to(device) for img in images_batch]
                targets_batch = [{k: v.to(device) for k, v in t.items()} for t in targets_batch]
                loss_dict = model(images_batch, targets_batch)
                losses = sum(loss for loss in loss_dict.values())
                val_loss += losses.item()
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "./pytorch/fasterrcnn_multiobject.pth")

    if plot:
        fig, ax = plt.subplots()
        ax.clear()
        ax.plot(range(1, epochs+1), train_losses, label='Train Loss', marker='o')
        ax.plot(range(1, epochs+1), val_losses, label='Val Loss', marker='o')
        ax.legend()
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.grid()
        fig.savefig("./fig/loss.png", dpi=300)
    return train_losses, val_losses

# 测试模型

In [ ]:
def test_model(test_loader, model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    images = []
    results = []
    all_targets = []

    with torch.no_grad():
        for images_batch, targets_batch in test_loader:
            images_batch = [img.to(device) for img in images_batch]
            outputs = model(images_batch)
            images.extend(images_batch)
            results.extend(outputs)
            all_targets.extend(targets_batch)

    return images, results, all_targets

# 使用模型进行预测

In [ ]:
def predict_model(model, image):
    image = normalization(image)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    with torch.no_grad():
        image_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0).repeat(3,1,1)
        image_tensor = image_tensor.to(device)
        outputs = model([image_tensor])

    return outputs[0]

# 开始使用模型进行训练、测试、预测
## 生成一些个训练数据

In [ ]:
# mun_simulation_datas = 5000
# images = []
# targets = []
# for i in tqdm(range(mun_simulation_datas), desc='generating', total=mun_simulation_datas):
#     image, target = simulation(320*120, 320*3, num_range=(10, 200), seed=i)
#     images.append(image)
#     targets.append(target)
#
# np.save('./data/simulation_data.npy', np.array(images))
# np.save('./data/simulation_targets.npy', targets, allow_pickle=True)

In [ ]:
images = np.load('../data/simulation_data.npy')
targets = np.load('../data/simulation_targets.npy', allow_pickle=True)

## 划分数据集

In [ ]:
train_loader, val_loader, test_loader = split_dataset(images=images, targets=targets)

## 训练模型

In [ ]:
train_loss, val_loss = train_model(train_loader, val_loader, epochs=50)

In [ ]:
def plot_detection(image, output, score_threshold=0.5):
    """
    在图像上绘制检测到的边界框。

    参数：
      - image: 原始图像，Tensor，形状为 [1, H, W]（灰度图）
      - output: 模型预测结果字典，包含 "boxes", "labels", "scores"
      - score_threshold: 仅显示置信度大于该阈值的检测结果
    """
    # 将图像转换为 numpy 数组，并将单通道转换为 2D 图像
    # image_np = image.squeeze(0).cpu().numpy().astype("float32")
    image_np = image

    # 创建图像绘制对象
    fig, ax = plt.subplots(1)
    ax.imshow(image_np, cmap="gray", origin="lower")

    # 获取预测边界框、置信度和标签
    boxes = output['boxes'].cpu().numpy()
    scores = output['scores'].cpu().numpy()
    labels = output['labels'].cpu().numpy()

    for box, score, label in zip(boxes, scores, labels):
        if score < score_threshold:
            continue  # 如果置信度低于阈值，则跳过该检测结果
        x_min, y_min, x_max, y_max = box
        width, height = x_max - x_min, y_max - y_min
        # 创建矩形补丁
        rect = patches.Rectangle((x_min, y_min), width, height, linewidth=0.5, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        # 添加置信度文字
        ax.text(x_min, y_min, f"{score:.2f}", color="yellow", fontsize=8)

    # for box, label in zip(boxes, labels):
    #     x_min, y_min, x_max, y_max = box
    #     width, height = x_max - x_min, y_max - y_min
    #     # 创建矩形补丁
    #     rect = patches.Rectangle((x_min, y_min), width, height, linewidth=0.5, edgecolor='red', facecolor='none')
    #     ax.add_patch(rect)

    plt.title("prediction")
    plt.show()

In [ ]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(num_classes=2, min_size=320, image_mean=[0, 0, 0], image_std=[1, 1, 1])
model.load_state_dict(torch.load("./pytorch/fasterrcnn_multiobject.pth", weights_only=True))

In [ ]:
val_images, results, real = test_model(test_loader, model)

In [ ]:
plot_detection(val_images[0][1, :, :], real[0], score_threshold=0.8)

In [17]:
predict = predict_model(model, np.abs(np.load('./工作总结/data/st_phasecong.npy')[0:320, 0:320]))
plot_detection(np.load('./工作总结/data/st_phasecong.npy')[0:320, 0:320], predict, score_threshold=0.5)

<IPython.core.display.Javascript object>

In [ ]:
np.save('./data/wave_label_predict.npy', predict, allow_pickle=True)